# Stage 1. Split train and test data

## 1. Setup

### 1.1 Imports

In [ ]:
import datetime
import logging
from pathlib import Path

from omegaconf import OmegaConf

from acid.config import load_config, prepare_output
from acid.data_preparation.format_str import format_series_str
from acid.data_preparation.map_category import map_fov_categories_df
from acid.data_preparation.train_test_split import add_train_test_split_clm
from acid.utils.metadata.loading import load_metadata

### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.dataset_splitting
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
metadata_df, metadata_file_name = load_metadata(stage_cfg.metadata)
plate_layout_df, plate_layout_file_name = load_metadata(stage_cfg.plate_layout)

## 3. Dataset splitting
### 3.1 Helpers

In [ ]:
# FUTURE: move to src/acid/data_preparation/map_category.py
def map_treatments(metadata_df, plate_layout_df, cfg):
    columns = cfg.metadata.dataframe_columns
    layout = plate_layout_df.copy()
    layout[columns.treatment_column_name] = format_series_str(
        layout[columns.treatment_column_name]
    )
    return map_fov_categories_df(
        metadata_df=metadata_df,
        plate__layout_df=layout,
        well__column=columns.well_column_name,
        experiment__column=columns.experiment_column_name,
        treatment__column=columns.treatment_column_name,
    )


# FUTURE: move to src/acid/data_preparation/train_test_split.py
def split_dataset(metadata_df, cfg):
    if cfg.processing.split_unit != "metadata_row":
        raise ValueError("Only metadata_row splitting is implemented")
    if not 0 < cfg.processing.test_fraction < 1:
        raise ValueError("test_fraction must be between zero and one")
    labels = cfg.dataset_split.labels
    if labels.train == labels.test:
        raise ValueError("Train and test labels must differ")
    return add_train_test_split_clm(
        df=metadata_df,
        test_size=cfg.processing.test_fraction,
        is_train_column=cfg.dataset_split.column,
        train_val=labels.train,
        test_val=labels.test,
        train_test_split_kwargs={"random_state": cfg.processing.random_state},
        concat_kwargs=OmegaConf.to_container(
            cfg.processing.concat_kwargs, resolve=True
        ),
    )

### 3.2 Map treatments and split

In [ ]:
metadata_df_w_treatment = map_treatments(metadata_df, plate_layout_df, stage_cfg)
metadata_df_w_split = split_dataset(metadata_df_w_treatment, stage_cfg)
metadata_df_w_split[stage_cfg.dataset_split.column].value_counts()

## 4. Save metadata

In [ ]:
# FUTURE: move to src/acid/utils/metadata/saving.py
def save_metadata_dataframe(metadata_df, metadata_config, project_name):
    saving = metadata_config.saving
    separator = saving.save_file_name_separator
    suffix = saving.metadata_file_suffix.format(save_file_name_separator=separator)
    filename = separator.join(
        [
            datetime.datetime.now().astimezone().strftime(saving.metadata_date_format),
            project_name,
            saving.metadata_savingword,
            suffix,
        ]
    )
    path = Path(metadata_config.directory) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata_df.to_csv(path, index=saving.save_csv_index)
    logging.getLogger(__name__).info("Saved metadata: %s", path)
    return path


metadata_path = save_metadata_dataframe(
    metadata_df_w_split, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
metadata_df_w_split.head()